In [1]:
import pandas as pd
import numpy as np
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN

In [2]:
# numpy needs to have the correct version!!
print("NumPy version:", np.__version__)


NumPy version: 1.26.4


# All texts including teacher interview

In [82]:
# Load the cleaned data
df = pd.read_csv('../outputs/cleaned_text_data.csv')

# For Option2 texts
option2_docs = df[df['religious_group'] == 'option2']['cleaned_content'].dropna()
option2_docs = option2_docs[option2_docs.str.strip() != '']
option2_docs = option2_docs.tolist()

# For Option1 texts
option1_docs = df[df['religious_group'] == 'option1']['cleaned_content'].dropna()
option1_docs = option1_docs[option1_docs.str.strip() != '']
option1_docs = option1_docs.tolist()

print(f"Option2 docs: {len(option2_docs)}")
print(f"Option1 docs: {len(option1_docs)}")

Option2 docs: 2795
Option1 docs: 1759


In [83]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer_model = CountVectorizer(stop_words="english", min_df=2, ngram_range=(1, 2))

In [84]:
import openai
from bertopic.representation import OpenAI
from dotenv import load_dotenv
import os

# Fine-tune topic representations with GPT
load_dotenv()
client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
openai_model = OpenAI(client, model="gpt-4o-mini", chat=True)

In [85]:
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, OpenAI

# KeyBERT
keybert_model = KeyBERTInspired()

# MMR
mmr_model = MaximalMarginalRelevance(diversity=0.3)


# All representation models
representation_model = {
    "KeyBERT": keybert_model,
    "OpenAI": openai_model,  # Uncomment if use OpenAI
    "MMR": mmr_model,
}

In [86]:
# OPTIMIZED BERTOPIC -  faster processing
print("Setting up optimized BERTopic models...")

# Use a faster, lighter embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Reduce UMAP dimensions for faster processing
umap_model = UMAP(n_neighbors=10, n_components=5, min_dist=0.0, metric='cosine', random_state=42)

# Optimize HDBSCAN for speed
hdbscan_model = HDBSCAN(min_cluster_size=5, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

# Create optimized BERTopic model for Option2
option2_topic_model_fast = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model, 
    hdbscan_model=hdbscan_model,
    language="english",
    calculate_probabilities=True,
    verbose=True,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model
)

print(f"Processing {len(option2_docs)} Option2 documents...")
option2_topics, option2_probs = option2_topic_model_fast.fit_transform(option2_docs)
print("Option2 topic modeling complete.")


Setting up optimized BERTopic models...


2025-09-09 11:28:25,293 - BERTopic - Embedding - Transforming documents to embeddings.


Processing 2795 Option2 documents...


Batches:   0%|          | 0/88 [00:00<?, ?it/s]

2025-09-09 11:28:32,292 - BERTopic - Embedding - Completed ✓
2025-09-09 11:28:32,293 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-09-09 11:28:39,505 - BERTopic - Dimensionality - Completed ✓
2025-09-09 11:28:39,506 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-09-09 11:28:40,423 - BERTopic - Cluster - Completed ✓
2025-09-09 11:28:40,425 - BERTopic - Representation - Fine-tuning topics using representation models.
100%|██████████| 143/143 [01:54<00:00,  1.25it/s]
2025-09-09 11:30:46,640 - BERTopic - Representation - Completed ✓


Option2 topic modeling complete.


In [87]:
# Create optimized BERTopic model for Option1
option1_topic_model_fast = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model, 
    hdbscan_model=hdbscan_model,
    language="english",
    calculate_probabilities=True,
    verbose=True,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model
)

print(f"Processing {len(option1_docs)} Option1 documents...")
option1_topics, option1_probs = option1_topic_model_fast.fit_transform(option1_docs)
print("Option1 topic modeling complete.")


2025-09-09 11:30:49,182 - BERTopic - Embedding - Transforming documents to embeddings.


Processing 1759 Option1 documents...


Batches:   0%|          | 0/55 [00:00<?, ?it/s]

2025-09-09 11:30:52,900 - BERTopic - Embedding - Completed ✓
2025-09-09 11:30:52,900 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-09-09 11:30:56,555 - BERTopic - Dimensionality - Completed ✓
2025-09-09 11:30:56,557 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-09-09 11:30:56,774 - BERTopic - Cluster - Completed ✓
2025-09-09 11:30:56,777 - BERTopic - Representation - Fine-tuning topics using representation models.
100%|██████████| 73/73 [00:59<00:00,  1.23it/s]
2025-09-09 11:32:02,252 - BERTopic - Representation - Completed ✓


Option1 topic modeling complete.


In [88]:
# save raw model
option1_topic_model_fast.save('../outputs/models/option1_topic_model_fast', serialization='safetensors')
option2_topic_model_fast.save('../outputs/models/option2_topic_model_fast', serialization='safetensors')

### additional details of the raw model

In [89]:
# Add top words as a column with proper error handling
def get_top_words(topic_num, topic_model):
    if topic_num == -1:  # Outlier topic
        return []
    topic_words = topic_model.get_topic(topic_num)
    if topic_words and isinstance(topic_words, list):
        return [word for word, score in topic_words[:10]]
    else:
        return []

In [90]:
# Create CSV files that EXACTLY match the topic modeling documents
import os

# Create output directory if it doesn't exist
os.makedirs('../outputs/analysis_results', exist_ok=True)

# IMPORTANT: Use the exact same filtering logic as Cell 3 to ensure perfect alignment
# Option2 - replicate exact filtering from Cell 3
option2_filtered_series = df[df['religious_group'] == 'option2']['cleaned_content'].dropna()
option2_filtered_series = option2_filtered_series[option2_filtered_series.str.strip() != '']
option2_valid_indices = option2_filtered_series.index  # Original indices in main df

# Option1 - replicate exact filtering from Cell 3
option1_filtered_series = df[df['religious_group'] == 'option1']['cleaned_content'].dropna()
option1_filtered_series = option1_filtered_series[option1_filtered_series.str.strip() != '']
option1_valid_indices = option1_filtered_series.index  # Original indices in main df

# Create dataframes using the EXACT same rows as topic modeling
option2_df = df.loc[option2_valid_indices].copy().reset_index(drop=True)
option1_df = df.loc[option1_valid_indices].copy().reset_index(drop=True)

# Save CSV files
option2_df.to_csv('../outputs/analysis_results/raw/option2/option2_documents.csv', index=False)
option1_df.to_csv('../outputs/analysis_results/raw/option1/option1_documents.csv', index=False)

In [91]:
# get raw content samples for each topic
option2_topic_info = option2_topic_model_fast.get_topic_info()

# Add top words as a column using lambda to pass the model
option2_topic_info['top_words'] = option2_topic_info['Topic'].apply(
    lambda topic_num: get_top_words(topic_num, option2_topic_model_fast)
)

def get_most_representative_samples(topic_num, topics, docs, probs, max_samples=3):
    """Get the most representative (highest confidence) text samples for a topic
    
    Returns:
        tuple: (samples, indices) where samples are the text content and indices are the original document indices
        For outlier topic (-1), returns empty lists
    """
    if topic_num == -1:  # Outlier topic - skip processing
        return [], []
    else:
        # Find all documents assigned to this topic
        topic_indices = [i for i, t in enumerate(topics) if t == topic_num]
        
        if len(topic_indices) == 0:
            return [], []
        
        # Get confidence scores for this topic
        topic_probs = [probs[i][topic_num] if topic_num < len(probs[i]) else 0 for i in topic_indices]
        
        # Get indices of most confident documents
        if len(topic_indices) <= max_samples:
            selected_indices = topic_indices
        else:
            # Sort by confidence and take top samples
            sorted_pairs = sorted(zip(topic_probs, topic_indices), reverse=True)
            selected_indices = [idx for _, idx in sorted_pairs[:max_samples]]
        
        samples = [docs[i] for i in selected_indices]
        return samples, selected_indices

# tuple return value for raw content samples and indices
def extract_samples_and_indices(topic_num):
    samples, indices = get_most_representative_samples(topic_num, option2_topics, option2_docs, option2_probs, max_samples=5)
    return samples

def extract_sample_indices(topic_num):
    samples, indices = get_most_representative_samples(topic_num, option2_topics, option2_docs, option2_probs, max_samples=5)
    return indices

option2_topic_info['raw_content_samples'] = option2_topic_info['Topic'].apply(extract_samples_and_indices)
option2_topic_info['sample_indices'] = option2_topic_info['Topic'].apply(extract_sample_indices)


In [92]:
# Save to CSV
option2_topic_info.to_csv('../outputs/analysis_results/raw/option2/option2_topic_info_detailed_raw.csv', index=False)
print(f"Option2 topic info saved. Found {len(option2_topic_info)} topics.")

Option2 topic info saved. Found 143 topics.


In [93]:

option1_topic_info = option1_topic_model_fast.get_topic_info()

# Add top words as a column using lambda to pass the model
option1_topic_info['top_words'] = option1_topic_info['Topic'].apply(
    lambda topic_num: get_top_words(topic_num, option1_topic_model_fast)
)

# Helper functions for Option1 
def extract_samples_and_indices_option1(topic_num):
    samples, indices = get_most_representative_samples(topic_num, option1_topics, option1_docs, option1_probs, max_samples=5)
    return samples

def extract_sample_indices_option1(topic_num):
    samples, indices = get_most_representative_samples(topic_num, option1_topics, option1_docs, option1_probs, max_samples=5)
    return indices

# Add raw content samples and their indices
option1_topic_info['raw_content_samples'] = option1_topic_info['Topic'].apply(extract_samples_and_indices_option1)
option1_topic_info['sample_indices'] = option1_topic_info['Topic'].apply(extract_sample_indices_option1)

In [94]:
# Save to CSV
option1_topic_info.to_csv('../outputs/analysis_results/raw/option1/option1_topic_info_detailed_raw.csv', index=False)
print(f"Option1 topic info saved. Found {len(option1_topic_info)} topics.")

Option1 topic info saved. Found 73 topics.


## fine-tune topics

### reduce outliers for option 1, raw

#### NOTE, this step directly updates the raw model! (save before proceed).

In [95]:
# Reduce outliers for Option1 using BERTopic's reduce_outliers method
print("Reducing outliers for Option1...")
print(f"Before: {sum(1 for t in option1_topics if t == -1)} outliers out of {len(option1_topics)} documents")

# Use BERTopic's reduce_outliers method
new_topics_option1 = option1_topic_model_fast.reduce_outliers(option1_docs, option1_topics, 
                                                             probabilities=option1_probs,
                                                             threshold=0.05, 
                                                             strategy="probabilities")

# Update the model with new topic assignments
option1_topic_model_fast.update_topics(option1_docs, topics=new_topics_option1)

print(f"After: {sum(1 for t in new_topics_option1 if t == -1)} outliers out of {len(new_topics_option1)} documents")
print(f"Reduced outliers by {sum(1 for t in option1_topics if t == -1) - sum(1 for t in new_topics_option1 if t == -1)} documents")

2025-09-09 11:33:06,895 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Reducing outliers for Option1...
Before: 341 outliers out of 1759 documents
After: 190 outliers out of 1759 documents
Reduced outliers by 151 documents


In [96]:
# Save the updated Option1 topic information and representations (CSV format)
print("Saving updated Option1 topic information with representations...")

# Get the updated topic info from the model  
option1_topic_info_updated = option1_topic_model_fast.get_topic_info()

# Add top words as a column using lambda to pass the model
option1_topic_info_updated['top_words'] = option1_topic_info_updated['Topic'].apply(
    lambda topic_num: get_top_words(topic_num, option1_topic_model_fast)
)

# Helper functions for updated topics (using new_topics_option1)
def extract_samples_and_indices_updated_option1(topic_num):
    samples, indices = get_most_representative_samples(topic_num, new_topics_option1, option1_docs, option1_probs, max_samples=5)
    return samples

def extract_sample_indices_updated_option1(topic_num):
    samples, indices = get_most_representative_samples(topic_num, new_topics_option1, option1_docs, option1_probs, max_samples=5)
    return indices

# Add raw content samples and their indices (using UPDATED topic assignments)
option1_topic_info_updated['raw_content_samples'] = option1_topic_info_updated['Topic'].apply(extract_samples_and_indices_updated_option1)
option1_topic_info_updated['sample_indices'] = option1_topic_info_updated['Topic'].apply(extract_sample_indices_updated_option1)

# Save to CSV
option1_topic_info_updated.to_csv('../outputs/analysis_results/raw/reduce_outlier/option1/option1_topic_info_reduced_outliers.csv', index=False)


Saving updated Option1 topic information with representations...


In [97]:
option1_topic_model_fast.save('../outputs/models/option1_topic_model_fast_reduced_outliers', serialization='safetensors')


### reduce outliers for option 2, raw


In [98]:
# Reduce outliers for Option2 using BERTopic's reduce_outliers method
print("Reducing outliers for Option2...")
print(f"Before: {sum(1 for t in option2_topics if t == -1)} outliers out of {len(option2_topics)} documents")

# Use BERTopic's reduce_outliers method
new_topics_option2 = option2_topic_model_fast.reduce_outliers(option2_docs, option2_topics, 
                                                             probabilities=option2_probs,
                                                             threshold=0.05, 
                                                             strategy="probabilities")

# Update the model with new topic assignments
option2_topic_model_fast.update_topics(option2_docs, topics=new_topics_option2)

print(f"After: {sum(1 for t in new_topics_option2 if t == -1)} outliers out of {len(new_topics_option2)} documents")
print(f"Reduced outliers by {sum(1 for t in option2_topics if t == -1) - sum(1 for t in new_topics_option2 if t == -1)} documents")


2025-09-09 11:33:21,448 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Reducing outliers for Option2...
Before: 718 outliers out of 2795 documents
After: 543 outliers out of 2795 documents
Reduced outliers by 175 documents


In [99]:
# Save the updated Option2 topic information and representations (CSV format)
print("Saving updated Option2 topic information with representations...")

# Get the updated topic info from the model  
option2_topic_info_updated = option2_topic_model_fast.get_topic_info()

# Add top words as a column using lambda to pass the model
option2_topic_info_updated['top_words'] = option2_topic_info_updated['Topic'].apply(
    lambda topic_num: get_top_words(topic_num, option2_topic_model_fast)
)

# Helper functions for updated topics (using new_topics_option2)
def extract_samples_and_indices_updated_option2(topic_num):
    samples, indices = get_most_representative_samples(topic_num, new_topics_option2, option2_docs, option2_probs, max_samples=5)
    return samples

def extract_sample_indices_updated_option2(topic_num):
    samples, indices = get_most_representative_samples(topic_num, new_topics_option2, option2_docs, option2_probs, max_samples=5)
    return indices

# Add raw content samples and their indices (using UPDATED topic assignments)
option2_topic_info_updated['raw_content_samples'] = option2_topic_info_updated['Topic'].apply(extract_samples_and_indices_updated_option2)
option2_topic_info_updated['sample_indices'] = option2_topic_info_updated['Topic'].apply(extract_sample_indices_updated_option2)

# Save to CSV
option2_topic_info_updated.to_csv('../outputs/analysis_results/raw/reduce_outlier/option2/option2_topic_info_reduced_outliers.csv', index=False)


Saving updated Option2 topic information with representations...


In [100]:
option2_topic_model_fast.save('../outputs/models/option2_topic_model_fast_reduced_outliers', serialization='safetensors')

### All texts visualizations 

note this is also based on reduced outliers model.

#### topics

In [101]:
fig1 = option1_topic_model_fast.visualize_topics()
fig1.write_html("../outputs/analysis_results/raw/reduce_outlier/visuals/option1_topics_visualization_reduce_outliers.html")

In [102]:
fig2 = option2_topic_model_fast.visualize_topics()
fig2.write_html('../outputs/analysis_results/raw/reduce_outlier/visuals/option2_topic_model_visualization_reduced_outlier.html')

In [103]:
# Option1 heatmap with clustering
print("Creating Option1 topic similarity heatmap...")
option1_heatmap = option1_topic_model_fast.visualize_heatmap(
    n_clusters=4,  # Number of topic clusters to form
)
option1_heatmap.write_html("../outputs/analysis_results/raw/reduce_outlier/visuals/option1_topic_heatmap_reduced_outliers.html")

Creating Option1 topic similarity heatmap...


In [104]:
# Create topic similarity heatmaps with clustering

# Option2 heatmap with clustering
print("Creating Option2 topic similarity heatmap...")
option2_heatmap = option2_topic_model_fast.visualize_heatmap(
    n_clusters=5,  # Number of topic clusters to form
)
option2_heatmap.write_html("../outputs/analysis_results/raw/reduce_outlier/visuals/option2_topic_heatmap.html")

Creating Option2 topic similarity heatmap...


#### documents

In [105]:
# Interactive Document Visualizations
# Following BERTopic documentation pipeline for speed optimization

print("Creating FAST interactive document visualizations...")
print("Step 1: Generating embeddings for documents...")

from sentence_transformers import SentenceTransformer
from umap import UMAP

# Use the same embedding model as our BERTopic (for consistency)
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate embeddings for Option2 documents  
print("Encoding Option2 documents...")
option2_embeddings = sentence_model.encode(option2_docs, show_progress_bar=True)

print("Step 2: Reducing embeddings to 2D for fast visualization...")
# Pre-reduce embeddings to 2D (much faster for iterative visualization)
reducer = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='cosine', random_state=42)
option2_reduced_embeddings = reducer.fit_transform(option2_embeddings)

print(f"Option2: Generated {len(option2_embeddings)} embeddings and reduced to 2D")

Creating FAST interactive document visualizations...
Step 1: Generating embeddings for documents...
Encoding Option2 documents...


Batches:   0%|          | 0/88 [00:00<?, ?it/s]

Step 2: Reducing embeddings to 2D for fast visualization...
Option2: Generated 2795 embeddings and reduced to 2D


In [106]:
# Generate embeddings for Option1 documents
print("Encoding Option1 documents...")
option1_embeddings = sentence_model.encode(option1_docs, show_progress_bar=True)

print("Reducing Option1 embeddings to 2D...")
# Use the same reducer configuration for consistency
reducer1 = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='cosine', random_state=42)
option1_reduced_embeddings = reducer1.fit_transform(option1_embeddings)

print(f"Option1: Generated {len(option1_embeddings)} embeddings and reduced to 2D")

Encoding Option1 documents...


Batches:   0%|          | 0/55 [00:00<?, ?it/s]

Reducing Option1 embeddings to 2D...
Option1: Generated 1759 embeddings and reduced to 2D


In [107]:
# Create FAST interactive DataMapPlots using pre-computed embeddings
print("Step 3: Creating interactive DataMapPlots with pre-computed embeddings...")

try:
    print("Creating Option2 interactive DataMapPlot...")
    # Use pre-computed reduced embeddings (MUCH faster!)
    option2_datamap = option2_topic_model_fast.visualize_document_datamap(
        option2_docs, 
        reduced_embeddings=option2_reduced_embeddings,  # Pre-computed!
        interactive=True,
        title="Option2 Documents Topic Map"
    )
    
    # Save as HTML file
    option2_datamap.save("../outputs/analysis_results/raw/reduce_outlier/visuals/option2_interactive_datamap_reduced_outliers.html")
    print("Option2 interactive DataMapPlot saved.")
    
except Exception as e:
    print(f"Error creating Option2 DataMapPlot: {e}")

try:
    print("Creating Option1 interactive DataMapPlot...")
    option1_datamap = option1_topic_model_fast.visualize_document_datamap(
        option1_docs, 
        reduced_embeddings=option1_reduced_embeddings,  # Pre-computed!
        interactive=True,
        title="Option1 Documents Topic Map"
    )
    
    # Save as HTML file
    option1_datamap.save("../outputs/analysis_results/raw/reduce_outlier/visuals/option1_interactive_datamap_reduced_outliers.html")
    print("Option1 interactive DataMapPlot saved.")
    
except Exception as e:
    print(f" Error creating Option1 DataMapPlot: {e}")


Step 3: Creating interactive DataMapPlots with pre-computed embeddings...
Creating Option2 interactive DataMapPlot...
Option2 interactive DataMapPlot saved.
Creating Option1 interactive DataMapPlot...
Option1 interactive DataMapPlot saved.


#### hierarchy

In [108]:
hierarchical_topics_option1 = option1_topic_model_fast.hierarchical_topics(option1_docs)

100%|██████████| 71/71 [00:00<00:00, 702.17it/s]


In [109]:
# Visualize the hierarchical topics
print("Creating hierarchical topics visualization...")

# Use the hierarchical_model to create the visualization
hierarchy_fig_option1 = option1_topic_model_fast.visualize_hierarchy(hierarchical_topics=hierarchical_topics_option1)
hierarchy_fig_option1.show()

# Also save as HTML
hierarchy_fig_option1.write_html("../outputs/analysis_results/raw/reduce_outlier/visuals/option1_hierarchical_topics_reduced_outliers.html")
print("Hierarchical topics visualization saved for option1")

Creating hierarchical topics visualization...


Hierarchical topics visualization saved for option1


In [110]:
hierarchical_topics_option2 = option2_topic_model_fast.hierarchical_topics(option2_docs)

100%|██████████| 141/141 [00:00<00:00, 684.83it/s]


In [111]:
# Visualize the hierarchical topics
print("Creating hierarchical topics visualization...")

# Use the hierarchical_model to create the visualization
hierarchy_fig_option2 = option2_topic_model_fast.visualize_hierarchy(hierarchical_topics=hierarchical_topics_option2)
hierarchy_fig_option2.show()

# Also save as HTML
hierarchy_fig_option2.write_html("../outputs/analysis_results/raw/reduce_outlier/visuals/option2_hierarchical_topics_reduced_outliers.html")
print("Hierarchical topics visualization saved for option2")

Creating hierarchical topics visualization...


Hierarchical topics visualization saved for option2


# Removing teacher interviews

In [112]:
# Filter out interviews using file_type column 
print("Filtering out interview data for textbook-only analysis (using file_type)...")

# Load the Option1 and Option2 dataframes 
option1_df = pd.read_csv('../outputs/analysis_results/raw/option1/option1_documents.csv')
option2_df = pd.read_csv('../outputs/analysis_results/raw/option2/option2_documents.csv')

# Filter out interviews using the file_type column (much simpler!)
option1_textbooks = option1_df[option1_df['file_type'] != 'interview'].copy()
option2_textbooks = option2_df[option2_df['file_type'] != 'interview'].copy()

# Extract cleaned content for topic modeling
option1_docs_textbooks = option1_textbooks['cleaned_content'].tolist()
option2_docs_textbooks = option2_textbooks['cleaned_content'].tolist()

# Save the filtered dataframes for reference
option1_textbooks.to_csv('../outputs/analysis_results/no_interview/option1/option1_no_interviews.csv', index=False)
option2_textbooks.to_csv('../outputs/analysis_results/no_interview/option2/option2_no_interviews.csv', index=False)

Filtering out interview data for textbook-only analysis (using file_type)...


In [113]:
# Train Option1 textbook-only topic model
print("Training Option1 textbook-only topic model...")

# Initialize components (same as before)
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
umap_model = UMAP(n_neighbors=10, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=5, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
vectorizer_model = CountVectorizer(stop_words="english", min_df=2, ngram_range=(1, 2))

# Representation models
keybert_model = KeyBERTInspired()
mmr_model = MaximalMarginalRelevance(diversity=0.3)
representation_model = {
    "KeyBERT": keybert_model,
    "MMR": mmr_model,
    "OpenAI": openai_model
}

# Create Option1 textbook-only model
option1_textbook_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    language="english",
    calculate_probabilities=True,
    verbose=True,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model
)

option1_topics_textbooks, option1_probs_textbooks = option1_textbook_model.fit_transform(option1_docs_textbooks)


Training Option1 textbook-only topic model...


2025-09-09 11:34:50,374 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/45 [00:00<?, ?it/s]

2025-09-09 11:34:53,676 - BERTopic - Embedding - Completed ✓
2025-09-09 11:34:53,676 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-09-09 11:34:56,456 - BERTopic - Dimensionality - Completed ✓
2025-09-09 11:34:56,457 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-09-09 11:34:56,632 - BERTopic - Cluster - Completed ✓
2025-09-09 11:34:56,634 - BERTopic - Representation - Fine-tuning topics using representation models.
100%|██████████| 75/75 [01:09<00:00,  1.08it/s]
2025-09-09 11:36:11,453 - BERTopic - Representation - Completed ✓


In [114]:

# Get topic info and add representative samples with indices
option1_textbook_topic_info = option1_textbook_model.get_topic_info()

# Add top words column
option1_textbook_topic_info['top_words'] = option1_textbook_topic_info['Topic'].apply(
    lambda topic_num: get_top_words(topic_num, option1_textbook_model)
)

# Helper functions for textbook topics
def extract_samples_textbooks_option1(topic_num):
    samples, indices = get_most_representative_samples(topic_num, option1_topics_textbooks, option1_docs_textbooks, option1_probs_textbooks, max_samples=5)
    return samples

def extract_indices_textbooks_option1(topic_num):
    samples, indices = get_most_representative_samples(topic_num, option1_topics_textbooks, option1_docs_textbooks, option1_probs_textbooks, max_samples=5)
    return indices

# Add representative samples and indices
option1_textbook_topic_info['raw_content_samples'] = option1_textbook_topic_info['Topic'].apply(extract_samples_textbooks_option1)
option1_textbook_topic_info['sample_indices'] = option1_textbook_topic_info['Topic'].apply(extract_indices_textbooks_option1)

# Save to CSV
option1_textbook_topic_info.to_csv('../outputs/analysis_results/no_interview/option1/option1_topic_info_no_interviews.csv', index=False)


In [115]:
# Train Option2 textbook-only topic model
print("Training Option2 textbook-only topic model...")

# Create new BERTopic model for Option2 textbooks only (using same settings)
option2_textbook_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    language="english",
    calculate_probabilities=True,
    verbose=True,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model
)

option2_topics_textbooks, option2_probs_textbooks = option2_textbook_model.fit_transform(option2_docs_textbooks)

# Display topic info
option2_textbook_topic_info = option2_textbook_model.get_topic_info()

option2_textbook_topic_info.to_csv('../outputs/analysis_results/no_interview/option2/option2_topic_info_no_interviews.csv', index=False)

2025-09-09 11:36:18,526 - BERTopic - Embedding - Transforming documents to embeddings.


Training Option2 textbook-only topic model...


Batches:   0%|          | 0/78 [00:00<?, ?it/s]

2025-09-09 11:36:25,398 - BERTopic - Embedding - Completed ✓
2025-09-09 11:36:25,399 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-09-09 11:36:31,439 - BERTopic - Dimensionality - Completed ✓
2025-09-09 11:36:31,440 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-09-09 11:36:32,274 - BERTopic - Cluster - Completed ✓
2025-09-09 11:36:32,277 - BERTopic - Representation - Fine-tuning topics using representation models.
100%|██████████| 137/137 [02:03<00:00,  1.11it/s]
2025-09-09 11:38:47,263 - BERTopic - Representation - Completed ✓


In [116]:
# Get topic info and add representative samples with indices
option2_textbook_topic_info = option2_textbook_model.get_topic_info()

# Add top words column
option2_textbook_topic_info['top_words'] = option2_textbook_topic_info['Topic'].apply(
    lambda topic_num: get_top_words(topic_num, option2_textbook_model)
)

# Helper functions for textbook topics
def extract_samples_textbooks_option2(topic_num):
    samples, indices = get_most_representative_samples(topic_num, option2_topics_textbooks, option2_docs_textbooks, option2_probs_textbooks, max_samples=5)
    return samples

def extract_indices_textbooks_option2(topic_num):
    samples, indices = get_most_representative_samples(topic_num, option2_topics_textbooks, option2_docs_textbooks, option2_probs_textbooks, max_samples=5)
    return indices

# Add representative samples and indices
option2_textbook_topic_info['raw_content_samples'] = option2_textbook_topic_info['Topic'].apply(extract_samples_textbooks_option2)
option2_textbook_topic_info['sample_indices'] = option2_textbook_topic_info['Topic'].apply(extract_indices_textbooks_option2)

# Save to CSV
option2_textbook_topic_info.to_csv('../outputs/analysis_results/no_interview/option2/option2_topic_info_no_interviews.csv', index=False)

In [117]:
# save the textbook models with safetensors
option1_textbook_model.save('../outputs/models/option1_textbook_model', serialization='safetensors')
option2_textbook_model.save('../outputs/models/option2_textbook_model', serialization='safetensors')

#### reduce outliers for the textbook model

In [118]:
# Use BERTopic's reduce_outliers method
new_topics_option2_textbook = option2_textbook_model.reduce_outliers(option2_docs_textbooks, option2_topics_textbooks, 
                                                             probabilities=option2_probs_textbooks,
                                                             threshold=0.05, 
                                                             strategy="probabilities")

# Update the model with new topic assignments
option2_textbook_model.update_topics(option2_docs_textbooks, topics=new_topics_option2_textbook)

2025-09-09 11:38:58,922 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


In [119]:
# Save the updated Option2 topic information and representations (CSV format)
print("Saving updated Option2 textbook topic information with representations...")

# Get the updated topic info from the model  
option2_textbook_topic_info_updated = option2_textbook_model.get_topic_info()

# Add top words as a column using lambda to pass the model
option2_textbook_topic_info_updated['top_words'] = option2_textbook_topic_info_updated['Topic'].apply(
    lambda topic_num: get_top_words(topic_num, option2_textbook_model)
)

# Helper functions for updated topics (using new_topics_option2)
def extract_samples_and_indices_updated_option2(topic_num):
    samples, indices = get_most_representative_samples(topic_num, new_topics_option2_textbook, option2_docs_textbooks, option2_probs_textbooks, max_samples=5)
    return samples

def extract_sample_indices_updated_option2(topic_num):
    samples, indices = get_most_representative_samples(topic_num, new_topics_option2_textbook, option2_docs_textbooks, option2_probs_textbooks, max_samples=5)
    return indices

# Add raw content samples and their indices (using UPDATED topic assignments)
option2_textbook_topic_info_updated['raw_content_samples'] = option2_textbook_topic_info_updated['Topic'].apply(extract_samples_and_indices_updated_option2)
option2_textbook_topic_info_updated['sample_indices'] = option2_textbook_topic_info_updated['Topic'].apply(extract_sample_indices_updated_option2)

# Save to CSV
option2_textbook_topic_info_updated.to_csv('../outputs/analysis_results/no_interview/reduced_outliers/option2/option2_textbook_topic_info_reduced_outliers.csv', index=False)


Saving updated Option2 textbook topic information with representations...


In [120]:
# Use BERTopic's reduce_outliers method
new_topics_option1_textbook = option1_textbook_model.reduce_outliers(option1_docs_textbooks, option1_topics_textbooks, 
                                                             probabilities=option1_probs_textbooks,
                                                             threshold=0.05, 
                                                             strategy="probabilities")

# Update the model with new topic assignments
option1_textbook_model.update_topics(option1_docs_textbooks, topics=new_topics_option1_textbook)

2025-09-09 11:39:09,223 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


In [121]:
# Save the updated Option1 topic information and representations (CSV format)
print("Saving updated Option1 textbook topic information with representations...")

# Get the updated topic info from the model  
option1_textbook_topic_info_updated = option1_textbook_model.get_topic_info()

# Add top words as a column using lambda to pass the model
option1_textbook_topic_info_updated['top_words'] = option1_textbook_topic_info_updated['Topic'].apply(
    lambda topic_num: get_top_words(topic_num, option1_textbook_model)
)

# Helper functions for updated topics (using new_topics_option2)
def extract_samples_and_indices_updated_option1(topic_num):
    samples, indices = get_most_representative_samples(topic_num, new_topics_option1_textbook, option1_docs_textbooks, option1_probs_textbooks, max_samples=5)
    return samples

def extract_sample_indices_updated_option1(topic_num):
    samples, indices = get_most_representative_samples(topic_num, new_topics_option1_textbook, option1_docs_textbooks, option1_probs_textbooks, max_samples=5)
    return indices

# Add raw content samples and their indices (using UPDATED topic assignments)
option1_textbook_topic_info_updated['raw_content_samples'] = option1_textbook_topic_info_updated['Topic'].apply(extract_samples_and_indices_updated_option1)
option1_textbook_topic_info_updated['sample_indices'] = option1_textbook_topic_info_updated['Topic'].apply(extract_sample_indices_updated_option1)

# Save to CSV
option1_textbook_topic_info_updated.to_csv('../outputs/analysis_results/no_interview/reduced_outliers/option1/option1_textbook_topic_info_reduced_outliers.csv', index=False)


Saving updated Option1 textbook topic information with representations...


### compare outlier numbers for textbook model, before and after reducing outliers

In [122]:
# Read the CSV files
option1_original = pd.read_csv('../outputs/analysis_results/no_interview/option1/option1_topic_info_no_interviews.csv')
option1_reduced = pd.read_csv('../outputs/analysis_results/no_interview/reduced_outliers/option1/option1_textbook_topic_info_reduced_outliers.csv')
option2_original = pd.read_csv('../outputs/analysis_results/no_interview/option2/option2_topic_info_no_interviews.csv')
option2_reduced = pd.read_csv('../outputs/analysis_results/no_interview/reduced_outliers/option2/option2_textbook_topic_info_reduced_outliers.csv')

# Extract outlier counts (Topic -1)
option1_original_outliers = option1_original[option1_original['Topic'] == -1]['Count'].iloc[0]
option1_reduced_outliers = option1_reduced[option1_reduced['Topic'] == -1]['Count'].iloc[0]
option2_original_outliers = option2_original[option2_original['Topic'] == -1]['Count'].iloc[0]
option2_reduced_outliers = option2_reduced[option2_reduced['Topic'] == -1]['Count'].iloc[0]

# Print comparison
print("OPTION 1 OUTLIER COMPARISON:")
print(f"Original: {option1_original_outliers} outlier documents")
print(f"Reduced:  {option1_reduced_outliers} outlier documents")
print(f"Reduction: {option1_original_outliers - option1_reduced_outliers} documents ({((option1_original_outliers - option1_reduced_outliers) / option1_original_outliers * 100):.1f}%)")
print()

print("OPTION 2 OUTLIER COMPARISON:")
print(f"Original: {option2_original_outliers} outlier documents")
print(f"Reduced:  {option2_reduced_outliers} outlier documents")
print(f"Reduction: {option2_original_outliers - option2_reduced_outliers} documents ({((option2_original_outliers - option2_reduced_outliers) / option2_original_outliers * 100):.1f}%)")

OPTION 1 OUTLIER COMPARISON:
Original: 330 outlier documents
Reduced:  227 outlier documents
Reduction: 103 documents (31.2%)

OPTION 2 OUTLIER COMPARISON:
Original: 607 outlier documents
Reduced:  434 outlier documents
Reduction: 173 documents (28.5%)


### visualizations for the textbook model with reduced outliers. 

note, if run the code of reducing outliers above, the no interview topic models (`option1_textbook_model` and `option2_textbook_model`) will be updated with reduce outlier setup.

In [123]:
fig11 = option1_textbook_model.visualize_topics()
fig11.write_html("../outputs/analysis_results/no_interview/reduced_outliers/visualizations/option1/option1_textbook_reduced_outliers_topics_visualization.html")

In [124]:
fig12 = option2_textbook_model.visualize_topics()
fig12.write_html("../outputs/analysis_results/no_interview/reduced_outliers/visualizations/option2/option2_textbook_reduced_outliers_topics_visualization.html")

In [125]:
# Option1 heatmap with clustering
print("Creating Option1 textbook model topic similarity heatmap...")
option11_heatmap = option1_textbook_model.visualize_heatmap(
    n_clusters=4,  # Number of topic clusters to form
)
option11_heatmap.write_html("../outputs/analysis_results/no_interview/reduced_outliers/visualizations/option1/option1_textbook_reduced_outliers_topics_heatmap.html")

Creating Option1 textbook model topic similarity heatmap...


In [126]:
# Option2 heatmap with clustering
print("Creating Option2 textbook model topic similarity heatmap...")
option12_heatmap = option2_textbook_model.visualize_heatmap(
    n_clusters=4,  # Number of topic clusters to form
)
option12_heatmap.write_html("../outputs/analysis_results/no_interview/reduced_outliers/visualizations/option2/option2_textbook_reduced_outliers_topics_heatmap.html")

Creating Option2 textbook model topic similarity heatmap...


In [127]:
print("Creating FAST interactive document visualizations...")
print("Step 1: Generating embeddings for documents...")

# Use the same embedding model as our BERTopic (for consistency)
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate embeddings for Option2 documents  
print("Encoding Option2 textbook documents...")
option2_textbook_embeddings = sentence_model.encode(option2_docs_textbooks, show_progress_bar=True)

print("Step 2: Reducing embeddings to 2D for fast visualization...")
# Pre-reduce embeddings to 2D (much faster for iterative visualization)
reducer = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='cosine', random_state=42)
option2_reduced_embeddings = reducer.fit_transform(option2_textbook_embeddings)

# Fix: Change option2_embeddings to option2_textbook_embeddings
print(f"Option2: Generated {len(option2_textbook_embeddings)} embeddings and reduced to 2D")

Creating FAST interactive document visualizations...
Step 1: Generating embeddings for documents...
Encoding Option2 textbook documents...


Batches:   0%|          | 0/78 [00:00<?, ?it/s]

Step 2: Reducing embeddings to 2D for fast visualization...
Option2: Generated 2476 embeddings and reduced to 2D


In [128]:
# Generate embeddings for Option1 documents
print("Encoding Option1 textbook documents...")
option1_textbook_embeddings = sentence_model.encode(option1_docs_textbooks, show_progress_bar=True)

print("Reducing Option1 textbook embeddings to 2D...")
# Use the same reducer configuration for consistency
reducer1 = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='cosine', random_state=42)
option1_textbook_reduced_embeddings = reducer1.fit_transform(option1_textbook_embeddings)

print(f"Option1: Generated {len(option1_textbook_embeddings)} embeddings and reduced to 2D")

Encoding Option1 textbook documents...


Batches:   0%|          | 0/45 [00:00<?, ?it/s]

Reducing Option1 textbook embeddings to 2D...
Option1: Generated 1434 embeddings and reduced to 2D


In [129]:
# Create FAST interactive DataMapPlots using pre-computed embeddings
print("Step 3: Creating interactive textbookDataMapPlots with pre-computed embeddings...")

try:
    print("Creating Option2 interactive DataMapPlot...")
    # Use pre-computed reduced embeddings (MUCH faster!)
    option2_datamap = option2_textbook_model.visualize_document_datamap(
        option2_docs_textbooks, 
        reduced_embeddings=option2_reduced_embeddings,  # Pre-computed!
        interactive=True,
        title="Option2 Textbook Documents Topic Map"
    )
    
    # Save as HTML file
    option2_datamap.save("../outputs/analysis_results/no_interview/reduced_outliers/visualizations/option2/option2_textbook_interactive_datamap.html")

    
except Exception as e:
    print(f"Error creating Option2 DataMapPlot: {e}")

try:
    print("Creating Option1 textbook interactive DataMapPlot...")
    option1_datamap = option1_textbook_model.visualize_document_datamap(
        option1_docs_textbooks, 
        reduced_embeddings=option1_textbook_reduced_embeddings,  # Pre-computed!
        interactive=True,
        title="Option1 Textbook Documents Topic Map"
    )
    
    # Save as HTML file
    option1_datamap.save("../outputs/analysis_results/no_interview/reduced_outliers/visualizations/option1/option1_textbook_interactive_datamap.html")
  
except Exception as e:
    print(f" Error creating Option1 DataMapPlot: {e}")

Step 3: Creating interactive textbookDataMapPlots with pre-computed embeddings...
Creating Option2 interactive DataMapPlot...
Creating Option1 textbook interactive DataMapPlot...


In [130]:
hierarchical_textbook_topics_option1 = option1_textbook_model.hierarchical_topics(option1_docs_textbooks)

# Visualize the hierarchical topics
print("Creating hierarchical textbook topics visualization...")

# Use the hierarchical_model to create the visualization
hierarchy_fig_textbook_option1 = option1_textbook_model.visualize_hierarchy(hierarchical_topics=hierarchical_textbook_topics_option1)

# Also save as HTML
hierarchy_fig_textbook_option1.write_html("../outputs/analysis_results/no_interview/reduced_outliers/visualizations/option1/option1_textbook_hierarchical_topics.html")


100%|██████████| 73/73 [00:00<00:00, 348.40it/s]

Creating hierarchical textbook topics visualization...


In [131]:
hierarchical_textbook_topics_option2= option2_textbook_model.hierarchical_topics(option2_docs_textbooks)

# Visualize the hierarchical topics
print("Creating hierarchical textbook topics visualization...")

# Use the hierarchical_model to create the visualization
hierarchy_fig_textbook_option2 = option2_textbook_model.visualize_hierarchy(hierarchical_topics=hierarchical_textbook_topics_option2)

# Also save as HTML
hierarchy_fig_textbook_option2.write_html("../outputs/analysis_results/no_interview/reduced_outliers/visualizations/option2/option2_textbook_hierarchical_topics.html")

100%|██████████| 135/135 [00:00<00:00, 606.88it/s]


Creating hierarchical textbook topics visualization...
